## Section 0 - Drive Mount,Code Import, HF Login , OUT/Cache Dir

In [1]:

# Mount google Drive
import os
from google.colab import drive
drive.mount('/content/drive')

# Add Repo, Clone/pull 
REPO_URL = 'https://github.com/rahulkolayikkath/synthetic-data-pipeline.git'  
%cd /content
if not os.path.exists('/content/synthetic-data-pipeline'):
    !git clone $REPO_URL
%cd /content/synthetic-data-pipeline
!git pull --ff-only

# Code import to working dir
!pip install -q -e .

# OUT_Dir --> out_quick 
os.makedirs('/content/drive/MyDrive/indic_synth/out_quick', exist_ok=True)
OUT = '/content/drive/MyDrive/indic_synth/out_quick'

Mounted at /content/drive
/content
Cloning into 'synthetic-data-pipeline'...
remote: Enumerating objects: 203, done.
remote: Counting objects: 100% (203/203), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 203 (delta 51), reused 188 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (203/203), 5.51 MiB | 27.10 MiB/s, done.
Resolving deltas: 100% (51/51), done.
/content/synthetic-data-pipeline
Already up to date.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for indic-synth (pyproject.toml) ... done


In [2]:
# HF token (kept out of git). Needs: accepted Kathbath terms + accepted Gemma-3 license.
from getpass import getpass
from huggingface_hub import login
os.environ['HF_TOKEN'] = getpass('HF token:')
login(os.environ['HF_TOKEN'])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
# Add Cache Dir for Gemma (24gb) - only Run cell if you have enough G-drive storage
import os
os.environ["HF_HOME"] = "/content/drive/MyDrive/indic_synth/hf_cache"

## Section 1 - 3 (till TTS Generation)

In [4]:
# Install for all stages till tts generation
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 122.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 113.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 121.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 61.9 MB/s eta 0:00:00


In [5]:
!python scripts/run.py --config config.quick.yaml --stages data_acquisition

07:30:02 INFO    run | Pipeline start | out_dir=/content/drive/MyDrive/indic_synth/out_quick seed=1234
07:30:02 INFO    run | =========== stage: data_acquisition ===========
07:30:02 INFO    run | Acquisition config: {'source': 'hf', 'repo_id': 'ai4bharat/Kathbath', 'local_dir': 'fake_kathbath', 'languages': ['hindi', 'malayalam'], 'split': 'valid', 'speakers_per_language': 2, 'clips_per_speaker': 2, 'min_total_speakers': 4, 'gender_balance': True, 'ref_min_dur': 3.0, 'ref_max_dur': 15.0, 'seed': 1234, 'out_dir': '/content/drive/MyDrive/indic_synth/out_quick', 'hf_token': True, 'max_retries': 4, 'retry_backoff': 2.0, 'force_catalog': False}
07:30:02 INFO    run | Loading cached catalog: /content/drive/MyDrive/indic_synth/out_quick/catalog.parquet
07:30:05 INFO    run | Selected 8 clips across 4 speakers; saved /content/drive/MyDrive/indic_synth/out_quick/selection_manifest.jsonl
07:30:06 INFO    run | 8 selected, 8 already done, 0 to pull
07:30:06 INFO    run | Pull complete: {'downloa

In [6]:
!python scripts/run.py --config config.quick.yaml --stages audio_engineering

07:30:17 INFO    run | Pipeline start | out_dir=/content/drive/MyDrive/indic_synth/out_quick seed=1234
07:30:17 INFO    run | =========== stage: audio_engineering ===========
07:30:18 INFO    run | 8 downloaded clips, 8 already prepared, 0 to process
07:30:19 INFO    run | Prepare complete: {"stage": "audio_engineering", "elapsed_sec": 0.0, "target_sr": 24000, "norm": "peak", "trim": false, "prepared": 0, "failed": 0, "sr_in": {}, "flags": {}, "backends": {}, "prepared_manifest": "/content/drive/MyDrive/indic_synth/out_quick/prepared_manifest.jsonl"}
07:30:19 INFO    run | Pipeline done in 2.2s. Final dataset: /content/drive/MyDrive/indic_synth/out_quick/dataset_manifest.jsonl


In [7]:
!python scripts/run.py --config config.quick.yaml --stages sentence_generation

07:30:23 INFO    run | Pipeline start | out_dir=/content/drive/MyDrive/indic_synth/out_quick seed=1234
07:30:23 INFO    run | =========== stage: sentence_generation ===========
Fetching 5 files:   0% 0/5 [00:00<?, ?it/s]
Fetching 5 files: 100% 5/5 [00:00<00:00,  5.72it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights:   0% 1/1065 [00:45<13:27:42, 45.55s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 1065/1065 [09:27<00:00,  1.88it/s] 
07:40:53 INFO    run | Model loaded: google/gemma-3-12b-it
Loading weights: 100% 199/199 [00:10<00:00, 19.83it/s]
07:42:25 INFO    run | Resumed 12 sentences from checkpoint.
07:42:25 INFO    run | Grid: 4 cells x quota 3 (~12 target sentences); already have 12.
07:42:25 INFO    run | [1/4] hi|Daily Commu

### Section 4 - TTS Generation

In [8]:
!pip install -q git+https://github.com/ai4bharat/IndicF5.git
!pip install -q "transformers==4.49.0" torch torchaudio soundfile speechbrain jiwer pyyaml

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.0/103.0 kB 11.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 105.2 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 94.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.0/113.0 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 k

In [ ]:
# Restart The session and Run Section 0 

In [4]:
# Check transformer Version  # -> 4.49.0
import transformers
print(transformers.__version__)

4.49.0


In [5]:
!python scripts/run.py --config config.quick.yaml --stages tts_generation

08:21:26 INFO    run | Pipeline start | out_dir=/content/drive/MyDrive/indic_synth/out_quick seed=1234
08:21:26 INFO    run | =========== stage: tts_generation ===========
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
2026-06-12 08:21:43.464555: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical oper

## Section - 5 QC 

In [4]:
# Update the Versions back
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 36.9 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 89.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 56.4 MB/s eta 0:00:00


In [ ]:
# Restart and Run Section zero

In [ ]:
!python scripts/run.py --config config.quick.yaml --stages quality_control

09:17:30 INFO    run | Pipeline start | out_dir=/content/drive/MyDrive/indic_synth/out_quick seed=1234
09:17:30 INFO    run | =========== stage: quality_control ===========
09:17:30 INFO    run | 12 utterances, 0 already QC'd, 12 to check
Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Fetching 404 files:   0% 0/404 [00:00<?, ?it/s]
Fetching 404 files:   0% 1/404 [00:00<02:55,  2.30it/s]
Fetching 404 files:   1% 3/404 [00:01<03:42,  1.80it/s]
Fetching 404 files:   3% 11/404 [00:01<00:51,  7.58it/s]
Fetching 404 files:   4% 16/404 [00:01<00:33, 11.59it/s]
Fetching 404 files:   5% 19/404 [00:02<00:32, 11.91it/s]
Fetching 404 files:   5% 22/404 [00:02<00:28, 13.64it/s]
Fetching 404 files:   6% 25/404 [00:02<00:25, 15.07it/s]
Fetching 404 files:   7% 28/404 [00:02<00:26, 14.23it/s]
Fetching 404 files:   8% 33/404 [00:02<00:22, 16.52it/s]
Fetching 404 files:   9% 35/404 [00:03<00:24, 15.07it/s]
Fetching 404 fi

In [9]:
print(open(f'{OUT}/qc_summary.json').read())

{
  "stage": "quality_control",
  "elapsed_sec": 606.89,
  "thresholds": {
    "cer_max": 0.15,
    "spk_cos_min": 0.6,
    "dur_per_char": [
      0.04,
      0.3
    ]
  },
  "checked": 12,
  "passed": 9,
  "failed": 3,
  "pass_rate": 0.75,
  "dataset_manifest": "/content/drive/MyDrive/indic_synth/out_quick/dataset_manifest.jsonl"
}


### Sanity Check - Rejected by QC Review

In [10]:
import json
import os
from typing import Dict, Iterable, Iterator, List
def read_jsonl(path: str) -> List[Dict]:
    """Read a JSONL file into a list of dicts. Missing file -> empty list."""
    if not os.path.exists(path):
        return []
    rows: List[Dict] = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

In [11]:
# sanity-check thresholds: listen to a couple of QC failures
from IPython.display import Audio, display
rows = read_jsonl(f'{OUT}/dataset_manifest.jsonl')
fails = [r for r in rows if not r['qc_passed']][:] # Edit show much you want to Verify 
print('pass:', sum(r['qc_passed'] for r in rows), '/', len(rows))
for r in fails:
    print(r['utt_id'], 'CER=', r.get('cer'), 'spk=', r.get('speaker_sim'), r['qc_reasons'])
    display(Audio(f"{OUT}/{r['audio_filepath']}"))

pass: 9 / 12
hi_000001 CER= 0.0 spk= 0.7327592968940735 ['truncation(tail/peak=0.25 (>0.25 => suspect cut-off))']


ml_000009 CER= 0.0 spk= 0.5395101308822632 ['speaker_cos(cos=0.540 (min 0.6))']


ml_000010 CER= 0.05405405405405406 spk= 0.8545917272567749 ['truncation(tail/peak=0.37 (>0.25 => suspect cut-off))']
